In [1]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset.csv")

/tmp/ipykernel_777670/20200156.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset.csv")


In [2]:
dataset.drop_duplicates(subset="SMILES").to_csv("unique_compounds.csv", index=False)

In [ ]:
import os
from deepmol.loaders import CSVLoader
from deepmol.compound_featurization import ThreeDimensionalMoleculeGenerator

# Processing parameters
timeout = 200
threads = 50
n_conformations = 1
max_iterations = 100
etkdg_version = 3
mode = "MMFF94"

dataset = CSVLoader("unique_compounds.csv", id_field="Substrate ID", smiles_field="SMILES").create_dataset()
            
# Generate 3D conformers
generator = ThreeDimensionalMoleculeGenerator(
    timeout_per_molecule=timeout, threads=threads,
    n_conformations=n_conformations, max_iterations=max_iterations
)
generator.generate(dataset, etkdg_version=etkdg_version, mode=mode)

# Save as SDF
output_sdf_path = os.path.join("unique_compounds_conformers.sdf")
dataset.to_sdf(output_sdf_path)

In [1]:
from deepmol.loaders._utils import load_csv_file, load_sdf_file
import pandas as pd

mols = load_sdf_file("unique_compounds_conformers.sdf")
ids = []
for mol in mols:
    id_ = mol.GetProp("_ID")
    ids.append(id_)

df = pd.DataFrame({"ids":ids, "mols": mols})

/home/jcapela/miniconda3/envs/esp/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: File error: Bad input file unique_compounds_conformers.sdf

In [12]:
df = df.drop_duplicates(subset="ids")

In [13]:
from deepmol.datasets import SmilesDataset

dataset = SmilesDataset(smiles=df.mols, mols=df.mols, ids=df.ids)
dataset.to_sdf("unique_compounds_conformers.sdf")

In [14]:
from deepmol.loaders import SDFLoader

compounds_dataset = SDFLoader("unique_compounds_conformers.sdf", id_field="_ID").create_dataset()

[22:08:48] WARNING: not removing hydrogen atom without neighbors
[22:08:48] WARNING: not removing hydrogen atom with dummy atom neighbors
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 69 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 71 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 82 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 95 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 69 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambig

3905
3905


In [15]:
compounds_dataset.ids

array(['compound_aminotransferase_dataset 1',
       'compound_aminotransferase_dataset 2',
       'compound_aminotransferase_dataset 3', ..., 'C06026',
       'CHEBI:58677', 'CHEBI:10329'], dtype='<U43')

In [16]:
compounds_dataset.get_shape()

2025-04-11 22:09:04,124 — INFO — Mols_shape: (3905,)
2025-04-11 22:09:04,125 — INFO — Features_shape: None
2025-04-11 22:09:04,126 — INFO — Labels_shape: None


((3905,), None, None)

In [17]:
from deepmol.compound_featurization import All3DDescriptors, MixedFeaturizer, TwoDimensionDescriptors

MixedFeaturizer([All3DDescriptors(mandatory_generation_of_conformers=False), TwoDimensionDescriptors()]).featurize(compounds_dataset, inplace=True)

2025-04-11 22:11:13,460 — ERROR — Failed to featurize *C(=O)NC(CO[C@@H]1O[C@H](CO)[C@@H](O[C@@H]2O[C@H](CO)[C@H](O[C@@H]3O[C@H](CO)[C@H](O)[C@H](O[C@@H]4O[C@H](CO)[C@H](O)[C@H](O[C@]5(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O5)[C@H]4O)[C@H]3NC(C)=O)[C@H](O[C@]3(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O3)[C@H]2O)[C@H](O)[C@H]1O)[C@H](O)/C=C/CCCCCCCCCCCCC. Appending empty array
2025-04-11 22:11:13,486 — ERROR — Exception message: 
2025-04-11 22:11:13,994 — ERROR — Failed to featurize *C(=O)NC(CO[C@@H]1O[C@H](CO)[C@@H](O[C@@H]2O[C@H](CO)[C@H](O[C@@H]3O[C@H](CO)[C@H](O)[C@H](O[C@@H]4O[C@H](CO)[C@H](O)[C@H](O[C@]5(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O5)[C@H]4O)[C@H]3NC(C)=O)[C@H](O[C@]3(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)C(CO)O[C@]4(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O4)O3)[C@H]2O)[C@H](O)[C@H]1O)[C@H](O)/C=C/CCCCCCCCCCCCC. Appending empty array
2025-04-11 22:11:13,996 — ERROR — Exception messa

MixedFeaturizer: 100%|██████████| 3905/3905 [02:04<00:00, 31.30it/s]


In [19]:
compounds_dataset.get_shape()

2025-04-11 22:11:19,109 — INFO — Mols_shape: (3298,)
2025-04-11 22:11:19,110 — INFO — Features_shape: (3298, 849)
2025-04-11 22:11:19,110 — INFO — Labels_shape: None


((3298,), (3298, 849), None)

In [20]:
compounds_dataset.to_sdf(path="unique_compounds_with_features.sdf")

In [21]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset.csv")
dataset

/tmp/ipykernel_1262493/4037893231.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset.csv")


,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,pubchem_id,Validated,RHEA_ID,EC number,reaction_SMILES
0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
1,MDYVTLASHAVRQYAPDQIFTASQRAKADAAALGEDAVINATLGEC...,C(C(=O)O)N,0.0,A0A174XK40,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,C(C(=O)O)N,0.0,P04693,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,C(C(=O)O)N,0.0,A0A3N0RMA5,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,C(C(=O)O)N,0.0,P0A959,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1015426,MEPEAPRRRHTHQRGYLLTRNPHLNKDLAFTLEERQQLNIHGLLPP...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,0.0,P48163,CHEBI:30616,NaN,NaN,False,NaN,NaN,NaN
1015427,MEPEAPRRRHTHQRGYLLTRNPHLNKDLAFTLEERQQLNIHGLLPP...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,0.0,P48163,C00002,NaN,NaN,False,NaN,NaN,NaN
1015428,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,COC1=C(OC)C(=O)C(CC=C(C)C)=C(C)C1=O,0.0,Q3TIG7,CHEBI:16389,NaN,NaN,False,NaN,NaN,NaN
1015429,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,N[C@@H](CCC([O-])=N[C@@H](CS)C(O)=NCC(=O)O)C(=O)O,0.0,Q3TIG7,CHEBI:57925,NaN,NaN,False,NaN,NaN,NaN


In [22]:
dataset[dataset["Substrate ID"].isin(compounds_dataset.ids)].to_csv("augmented_dataset_descriptors_available.csv", index=False)

In [23]:
compounds_dataset.to_csv("unique_compounds_with_features.csv", index=False)

In [24]:
import pandas as pd
compounds_with_features = pd.read_csv("unique_compounds_with_features.csv")
compounds_with_features

,ids,smiles,Asphericity,AUTOCORR3D_0,AUTOCORR3D_1,AUTOCORR3D_2,AUTOCORR3D_3,AUTOCORR3D_4,AUTOCORR3D_5,AUTOCORR3D_6,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,compound_aminotransferase_dataset 1,NCC(=O)O,0.408650,0.557,0.955,0.648,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,compound_aminotransferase_dataset 2,C[C@H](N)C(=O)O,0.273338,0.474,0.970,0.863,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,compound_aminotransferase_dataset 3,CC(C)[C@H](N)C(=O)O,0.100952,0.364,0.795,0.935,0.521,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,compound_aminotransferase_dataset 4,CC(C)C[C@H](N)C(=O)O,0.442271,0.326,0.687,0.748,0.748,0.600,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,compound_aminotransferase_dataset 5,CC[C@H](C)[C@H](N)C(=O)O,0.278073,0.326,0.692,0.921,0.718,0.278,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3293,CHEBI:58158,[NH3+]C(C(=O)[O-])C(=O)[O-],0.215313,0.347,0.770,0.898,0.549,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3294,CHEBI:78435,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,0.920867,0.023,0.054,0.079,0.107,0.138,0.175,0.195,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3295,C15778,C=C(CC[C@@H](C)[C@H]1CC[C@H]2C3=CC=C4C[C@@H](O...,0.745251,0.120,0.312,0.487,0.552,0.558,0.595,0.618,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3296,CHEBI:58677,C[C@H](CCC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3[C@H](...,0.754632,0.041,0.106,0.157,0.203,0.233,0.247,0.281,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [25]:
from plants_sm.io.pickle import write_pickle

features_dict = compounds_with_features.set_index('ids').drop(columns='smiles').apply(lambda row: row.to_numpy(), axis=1).to_dict()
write_pickle("compounds_features.pkl", features_dict)

True

In [26]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")
dataset["Enzyme ID"] = dataset["Enzyme ID"].str.replace(" ", "_")
dataset.to_csv("augmented_dataset_descriptors_available.csv", index=False)

/tmp/ipykernel_1262493/982059309.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")


In [27]:
dataset.Binding.value_counts()

0.0    518892
1.0    301029
Name: Binding, dtype: int64

In [28]:
dataset.sample(frac=0.1).to_csv("test_integrated_dataset.csv")

In [29]:
dataset.shape

(819921, 11)

In [30]:
dataset.drop_duplicates(subset=["Sequence"]).loc[:, ["Sequence", "Enzyme ID"]].to_csv("unique_enzymes.csv", index=False)

In [32]:
dataset.drop_duplicates(subset=["Sequence"]).loc[:, ["Sequence", "Enzyme ID"]]

,Sequence,Enzyme ID
0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,P00509
1,MDYVTLASHAVRQYAPDQIFTASQRAKADAAALGEDAVINATLGEC...,A0A174XK40
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,P04693
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,A0A3N0RMA5
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,P0A959
...,...,...
819797,MADSHSVNNQLSRHTSIFGLRLWVVLGVCVGAAIVLLLVLISLWFI...,O22764
819821,MNLKQFTCLSCAQLLAILLFIFAFFPRKIVLTGISKQDPDQDRDLQ...,P40367
819840,MKTETHNRQCIPLAISVLSLFINGVSSARQPPDRCNRVCGEISIPF...,Q7X8C5
819875,MGSSEPLPIVDSDKRRKKKRKTRATDSLPGKFEDVYQLTSELLGEG...,Q4G050


In [31]:
import csv

def csv_to_fasta(csv_file_path, fasta_file_path):
    with open(csv_file_path, mode='r', newline='') as csv_file, open(fasta_file_path, mode='w') as fasta_file:
        csv_reader = csv.reader(csv_file)
        next(csv_reader)  # Skip the header row if there is one

        for row in csv_reader:
            if len(row) >= 2:
                sequence_id = row[1].replace(" ", "_")
                sequence = row[0]
                fasta_file.write(f">{sequence_id}\n")
                fasta_file.write(f"{sequence}\n")

# Example usage
csv_file_path = 'unique_enzymes.csv'  # Path to your CSV file
fasta_file_path = 'unique_enzymes.fasta'  # Path to save the FASTA file
csv_to_fasta(csv_file_path, fasta_file_path)

In [1]:
import pandas as pd

# dataset = pd.read_csv("integrated_dataset_descriptors_available.csv")
from plants_sm.io.pickle import read_pickle
splits = read_pickle("splits/splits_0_6_proteins_train_val_test.pkl")


In [2]:
import pickle
# Save the pickle file with a compatible protocol version
with open('splits_0_6_proteins_train_val_test_4.pkl', 'wb') as f:
    pickle.dump(splits, f, protocol=4)  # Use protocol 4 for compatibility with Python 3.4+

In [19]:
train_dataset = dataset[dataset["Enzyme ID"].isin(splits[0][0])]
train_dataset

,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,pubchem_id
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,C(C(=O)O)N,0.0,P04693,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,C(C(=O)O)N,0.0,A0A3N0RMA5,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,C(C(=O)O)N,0.0,P0A959,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
5,MADTRPERRFTRIDRLPPYVFNITAELKMAARRRGEDIIDFSMGNP...,C(C(=O)O)N,0.0,P77434,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
9,MVITIDMLNPNVIETQYAVRGPIIILAQELEKKGKKMYYFNIGNPQ...,C(C(=O)O)N,0.0,A0A1Q9NYW4,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
...,...,...,...,...,...,...,...
74301,MKYSRVFINSLAYELALVVVSSSELESRLAPLYQKFRIPMGQLAAL...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Shewanella_putrefaciens,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0
74302,MVKTMNYIAKLVGVGQALPERVVTSEEMEAMMGFDQLGIRKGMAKI...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Anaerocolumna_xylanovorans,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0
74304,MAFLSVNNVEIVGLAAAVPKNVETLDNLEFFAPGEAEKVMALTGIK...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Bacteroides_luti,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0
74305,MSAPRYSQVSAVAVRLPDEDLTTPELEELLAERNPRVDVPRGLIER...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Nonomuraea_candida,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0


In [3]:
import pandas as pd

pd.read_csv("descriptors_np_classifier.csv").iloc[:, :12]

FileNotFoundError: [Errno 2] No such file or directory: 'descriptors_np_classifier.csv'

In [4]:
import pandas as pd

pd.read_csv("binding_np_classifier.csv").sort_values(by="mcc validation set", ascending=False)

,Model type,Trial,F1_macro_validation_set,Precision validation set,recall validation set,roc_auc validation set,accuracy validation set,mcc validation set,F1_macro_test_set,Precision test set,...,accuracy test set,mcc test set,Batch size,Learning rate,Protein layers,Compound layers,Final head layers,Epochs,batch_norm,dropout
46,gat,46,0.811786,0.947228,0.710232,0.898390,0.890600,0.752058,0.782297,0.885306,...,0.870655,0.701623,64,0.000799,"[2560, 1280, 1280, 1280]","[2560, 1280]","[2560, 1280, 1280]",135,True,0.789011
27,gat,27,0.794070,0.944353,0.685052,0.909235,0.881970,0.732333,0.756797,0.839867,...,0.853211,0.659744,64,0.000456,"[2560, 1280]","[2560, 1280]","[2560, 1280]",100,False,0.713421
28,gat,28,0.785033,0.972255,0.658273,0.902131,0.880244,0.732043,0.784341,0.950874,...,0.878279,0.724502,64,0.000761,"[2560, 2560, 2560, 1280]","[2560, 1280]","[2560, 1280]",97,True,0.603956
90,gat,90,0.796533,0.927736,0.697842,0.901046,0.881572,0.729775,0.784832,0.884062,...,0.871689,0.704018,64,0.000597,"[2560, 1280]","[2560, 1280]","[2560, 1280]",81,True,0.764954
97,gat,97,0.789437,0.950478,0.675060,0.909117,0.880377,0.729457,0.784301,0.920472,...,0.875371,0.714703,32,0.000721,"[2560, 1280]","[2560, 1280, 1280]","[2560, 1280]",165,True,0.779733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,gat,48,0.763787,0.755573,0.772182,0.897413,0.841344,0.644455,0.777396,0.783004,...,0.853405,0.668162,64,0.001072,"[2560, 1280]","[2560, 1280, 1280, 1280]","[2560, 1280, 1280]",148,True,0.760131
78,gat,78,0.746147,0.803226,0.696643,0.896220,0.842539,0.636224,0.775602,0.813958,...,0.857863,0.673531,64,0.000721,"[1280, 640]","[2560, 2560, 2560, 1280, 640]","[2560, 1280, 1280]",85,False,0.642645
1,gat,1,0.747532,0.711858,0.786970,0.896487,0.823420,0.614108,0.726841,0.737856,...,0.821489,0.594456,64,0.001077,"[1280, 640]","[2560, 1280]","[2560, 1280, 1280]",149,False,0.321981
17,gat,17,0.710643,0.603858,0.863309,0.896442,0.766463,0.548554,0.685697,0.569661,...,0.738209,0.506893,64,0.000108,"[2560, 1280, 1280, 1280, 640]","[2560, 1280, 1280, 1280, 640]","[1280, 640]",57,True,0.992224


In [5]:
import pandas as pd

pd.read_csv("binding_3d_gat_np_classifier.csv").sort_values(by="mcc validation set", ascending=False).iloc[:, 14:]

,Batch size,Learning rate,num_heads_gat,num_heads_attention,activation_attention,dropout_gat,num_heads_descriptors_cross_attention,prediction_layers,dropout_attention,patience
42,64,0.000238,2,4,ReLU(),0.5,5,[512],0.5,5
70,32,0.000403,1,1,ELU(alpha=1.0),0.5,5,"[2048, 1024]",0.2,5
61,32,0.000092,8,4,ReLU(),0.2,5,[512],0.2,5
96,64,0.000158,2,8,ReLU(),0.5,5,"[2048, 1024]",0.4,5
9,64,0.000425,8,8,Tanh(),0.2,5,"[2048, 1024, 1024]",0.2,5
...,...,...,...,...,...,...,...,...,...,...
26,16,0.000634,1,8,LeakyReLU(negative_slope=0.01),0.4,1,"[2048, 1024]",0.2,5
63,32,0.000079,8,4,ReLU(),0.2,5,"[2048, 1024]",0.2,5
76,32,0.000414,2,1,Tanh(),0.2,5,[512],0.5,5
8,32,0.002103,4,8,ELU(alpha=1.0),0.2,1,[128],0.5,5


In [3]:
import pandas as pd

pd.read_csv("binding_np_classifier_weighted_bce.csv").sort_values(by="mcc validation set", ascending=False).iloc[0, 14:]

Batch size                                      32
Learning rate                             0.000063
Protein layers                  [2560, 1280, 1280]
Compound layers      [2560, 2560, 2560, 1280, 640]
Final head layers    [2560, 2560, 2560, 1280, 640]
Epochs                                          80
batch_norm                                    True
dropout                                   0.596622
alpha                                         0.25
gamma                                          2.0
lambda_focal                                     0
Best epoch                                       1
Name: 10, dtype: object

In [6]:
protein_descriptors = read_pickle("propythia_descriptors/features.pkl")["proteins"]
compounds_features = read_pickle("compounds_features.pkl")

In [1]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")
dataset 

/tmp/ipykernel_2933046/672680221.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")


,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,pubchem_id,Validated,RHEA_ID,EC number,reaction_SMILES
0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
1,MDYVTLASHAVRQYAPDQIFTASQRAKADAAALGEDAVINATLGEC...,C(C(=O)O)N,0.0,A0A174XK40,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,C(C(=O)O)N,0.0,P04693,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,C(C(=O)O)N,0.0,A0A3N0RMA5,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,C(C(=O)O)N,0.0,P0A959,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
819916,MSEAEARPTNFIRQIIDEDLASGKHTTVHTRFPPEPNGYLHIGHAK...,OC[C@@H](O)[C@H](O)[C@@H](O)CO,0.0,P00962,CHEBI:17151,NaN,NaN,False,NaN,NaN,NaN
819917,MEPEAPRRRHTHQRGYLLTRNPHLNKDLAFTLEERQQLNIHGLLPP...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,0.0,P48163,CHEBI:30616,NaN,NaN,False,NaN,NaN,NaN
819918,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,COC1=C(OC)C(=O)C(CC=C(C)C)=C(C)C1=O,0.0,Q3TIG7,CHEBI:16389,NaN,NaN,False,NaN,NaN,NaN
819919,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,N[C@@H](CCC([O-])=N[C@@H](CS)C(O)=NCC(=O)O)C(=O)O,0.0,Q3TIG7,CHEBI:57925,NaN,NaN,False,NaN,NaN,NaN


In [3]:
dataset[dataset["Validated"]==True].to_csv("augmented_dataset_descriptors_available_validated.csv", index=False)